# Juvenile Mortality Modeling Notebook  

This notebook is rewritten to use **one source file only**:

```text
script/data/parquet/full.parquet
```

It is organized into four sessions:

1. **Data overview and filters**
2. **Model screening / feature selection**
3. **Model smoothing and model selection**
4. **Post-model testing / validation**

### Design principles
- self-contained: no `config.py`, no custom project modules
- easy to edit: the first parameter block is the control panel
- practical for a large parquet file
- count-focused for mortality model fitting, while still carrying amount fields for sensitivity review

### Important note on the 1M face-amount cap option
When `CAP_FACE_AT_1M = True`, the notebook applies exactly the rule you requested:

- source face bands `08 / 09 / 10 / 11` are collapsed into  
  `'08: 1,000,000+'`
- for those rows,  
  `Death_Claim_Amount = 1_000_000 × Death_Count`

This transformation is intentionally limited to:
- `Face_Amount_Band`
- `Death_Claim_Amount`

It does **not** alter:
- `Amount_Exposed`
- `ExpDth_VBT2015_Amt`

That keeps the rule transparent and easy to revise later if you decide to extend the cap logic.


In [2]:
from __future__ import annotations

import warnings
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from patsy import bs

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

try:
    import pyarrow  # noqa: F401
    HAS_PYARROW = True
except Exception:
    HAS_PYARROW = False

print("duckdb available :", HAS_DUCKDB)
print("pyarrow available:", HAS_PYARROW)


duckdb available : True
pyarrow available: True


---

# Session 1 · Data overview and filters

This section is the control panel for the whole notebook.

Edit the parameters here first. The defaults are aligned with the juvenile-scope choices you have discussed repeatedly:
- juvenile issue ages only
- unismoke only
- exclude insurance plan `Other`
- optional PLT exclusion
- optional 1M face-amount cap


In [3]:
# ============================================================
# CONTROL PANEL — EDIT HERE FIRST
# ============================================================

DATA_PATH = Path("script/data/parquet/full.parquet")

# ------------------------------
# Core scope filters
# ------------------------------
OBSERVATION_YEARS = None          # Example: [2018, 2019]; use None to keep all years
ISSUE_AGE_MIN = 0
ISSUE_AGE_MAX = 17
ATTAINED_AGE_MAX = 94

SEX_KEEP = None                   # Example: ["F", "M"]; None keeps all
SMOKER_STATUS_KEEP = ["U"]        # Example: ["U"]; set to None to keep all
INSURANCE_PLAN_DROP = ["Other"]   # Set to [] if you want to keep "Other"

EXCLUDE_POST_LEVEL_TERM = True
POST_LEVEL_TERM_EXCLUDE_VALUES = ["PLT"]   # Based on SOA_Post_Lvl_Ind

# Optional duration cleanup often used in reporting
COLLAPSE_DURATION_26_PLUS = False

# ------------------------------
# 1M face-amount cap option
# ------------------------------
CAP_FACE_AT_1M = True
FACE_CAP_SOURCE_BANDS = [
    "08: 1,000,000 - 2,499,999",
    "09: 2,500,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999",
    "11: 10,000,000+",
]
FACE_CAP_TARGET_BAND = "08: 1,000,000+"

# ------------------------------
# Modeling controls
# ------------------------------
ACTUAL_CNT_COL = "Death_Count"
EXPECTED_CNT_COL = "ExpDth_VBT2015_Cnt"
ACTUAL_AMT_COL = "Death_Claim_Amount"
EXPECTED_AMT_COL = "ExpDth_VBT2015_Amt"

NUMERIC_COLS = [
    "Observation_Year", "Issue_Age", "Duration", "Issue_Year", "Attained_Age",
    "Amount_Exposed", "Policies_Exposed", ACTUAL_AMT_COL, ACTUAL_CNT_COL,
    EXPECTED_CNT_COL, EXPECTED_AMT_COL
]

CATEGORICAL_COLS = [
    "Age_Ind", "Sex", "Smoker_Status", "Insurance_Plan", "Face_Amount_Band",
    "SOA_Antp_Lvl_TP", "SOA_Guar_Lvl_TP", "SOA_Post_Lvl_Ind", "Slct_Ult_Ind",
    "Preferred_Indicator", "Number_of_Pfd_Classes", "Preferred_Class"
]

BASE_COLUMNS = NUMERIC_COLS + CATEGORICAL_COLS

# Candidate features for screening.
# Comment out any feature you do not want screened.
SCREEN_FEATURES = [
    "Sex",
    "Insurance_Plan",
    "Face_Amount_Band",
    "Slct_Ult_Ind",
    "Age_Ind",
    "Preferred_Indicator",
    "Number_of_Pfd_Classes",
    "Issue_Age",
    "Attained_Age",
    "Duration",
    "Observation_Year",
    "SOA_Guar_Lvl_TP",
    "SOA_Antp_Lvl_TP",
    "SOA_Post_Lvl_Ind",
]

NUMERIC_SCREEN_FEATURES = ["Issue_Age", "Attained_Age", "Duration", "Observation_Year"]
SCREEN_SPLINE_DF = 6

# Manual shortlist after screening.
# Edit this after you review the Session 2 table.
SHORTLIST_FEATURES = [
    "Sex",
    "Insurance_Plan",
    "Face_Amount_Band",
    "Slct_Ult_Ind",
    "Issue_Age",
    "Duration",
]

# Holdout logic for model selection.
# If not None, these years are held out from fitting and used only for evaluation.
HOLDOUT_YEARS = [2018, 2019]

# Candidate models for Session 3.
# Keep these explicit so you can easily comment out / rewrite them.
CANDIDATE_MODELS = {
    "M1_issue_duration_core":
        f"{ACTUAL_CNT_COL} ~ C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
        f"+ C(Slct_Ult_Ind) + bs(Issue_Age, df=6, degree=3, include_intercept=False) "
        f"+ bs(Duration, df=6, degree=3, include_intercept=False)",

    "M2_issue_duration_plus_pref":
        f"{ACTUAL_CNT_COL} ~ C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
        f"+ C(Slct_Ult_Ind) + C(Preferred_Indicator) "
        f"+ bs(Issue_Age, df=6, degree=3, include_intercept=False) "
        f"+ bs(Duration, df=6, degree=3, include_intercept=False)",

    "M3_attained_age_core":
        f"{ACTUAL_CNT_COL} ~ C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
        f"+ C(Slct_Ult_Ind) + bs(Attained_Age, df=7, degree=3, include_intercept=False)",

    "M4_issue_duration_term":
        f"{ACTUAL_CNT_COL} ~ C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
        f"+ C(Slct_Ult_Ind) + C(SOA_Guar_Lvl_TP) "
        f"+ bs(Issue_Age, df=6, degree=3, include_intercept=False) "
        f"+ bs(Duration, df=6, degree=3, include_intercept=False)",
}

# Final model choice for Session 4.
# Set this after reviewing the Session 3 comparison table.
SELECTED_MODEL_KEY = "M1_issue_duration_core"

EXPORT_DIR = Path("script/model_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ============================================================
# DATA DICTIONARY USED BY THIS NOTEBOOK
# ============================================================

DATA_DICTIONARY = {
    "Observation_Year": "Calendar year of observation (2012-2019).",
    "Age_Ind": "0 = ANB, 1 = ALB, 2 = Age next birthday.",
    "Sex": "F = Female, M = Male.",
    "Smoker_Status": "N = Nonsmoker, S = Smoker, U = Uni-smoke.",
    "Insurance_Plan": "Perm, Term, UL, ULSG, VL, VLSG, Other.",
    "Issue_Age": "Issue age.",
    "Duration": "Policy duration.",
    "Face_Amount_Band": "Face amount band (01 to 11).",
    "Issue_Year": "Issue year.",
    "Attained_Age": "Attained age.",
    "SOA_Antp_Lvl_TP": "SOA anticipated level term period.",
    "SOA_Guar_Lvl_TP": "SOA guaranteed level term period.",
    "SOA_Post_Lvl_Ind": "SOA post level term indicator.",
    "Slct_Ult_Ind": "S = Select, U = Ultimate.",
    "Preferred_Indicator": "0 = Not preferred, 1 = Preferred, U = Unknown.",
    "Number_of_Pfd_Classes": "Number of preferred classes.",
    "Preferred_Class": "Preferred class code.",
    "Amount_Exposed": "Amount exposed.",
    "Policies_Exposed": "Policies exposed.",
    "Death_Claim_Amount": "Observed death claim amount.",
    "Death_Count": "Observed death count.",
    "ExpDth_VBT2015_Cnt": "Expected deaths on count basis under 2015 VBT.",
    "ExpDth_VBT2015_Amt": "Expected deaths on amount basis under 2015 VBT.",
}

display(pd.DataFrame(
    {"Column": list(DATA_DICTIONARY.keys()), "Description": list(DATA_DICTIONARY.values())}
))


,Column,Description
0,Observation_Year,Calendar year of observation (2012-2019).
1,Age_Ind,"0 = ANB, 1 = ALB, 2 = Age next birthday."
2,Sex,"F = Female, M = Male."
3,Smoker_Status,"N = Nonsmoker, S = Smoker, U = Uni-smoke."
4,Insurance_Plan,"Perm, Term, UL, ULSG, VL, VLSG, Other."
5,Issue_Age,Issue age.
6,Duration,Policy duration.
7,Face_Amount_Band,Face amount band (01 to 11).
8,Issue_Year,Issue year.
9,Attained_Age,Attained age.


In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def require_parquet_support():
    if HAS_DUCKDB or HAS_PYARROW:
        return
    raise ImportError(
        "This notebook needs either duckdb or pyarrow to read parquet. "
        "Install one of them locally, for example: pip install duckdb or pip install pyarrow"
    )


def read_parquet_data(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """
    Read the parquet file using duckdb if available, otherwise pandas/pyarrow.
    """
    require_parquet_support()

    if not path.exists():
        raise FileNotFoundError(f"Parquet file not found: {path}")

    if HAS_DUCKDB:
        cols_sql = "*" if columns is None else ", ".join([f'"{c}"' for c in columns])
        query = f"SELECT {cols_sql} FROM read_parquet('{path.as_posix()}')"
        return duckdb.sql(query).df()

    return pd.read_parquet(path, columns=columns)


def standardize_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in NUMERIC_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in CATEGORICAL_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("Missing")

    return out


def apply_face_amount_cap_1m(df: pd.DataFrame) -> pd.DataFrame:
    """
    Requested cap rule:
    - bands 08/09/10/11 -> '08: 1,000,000+'
    - Death_Claim_Amount = 1,000,000 * Death_Count
    """
    if not CAP_FACE_AT_1M:
        return df.copy()

    out = df.copy()
    mask = out["Face_Amount_Band"].isin(FACE_CAP_SOURCE_BANDS)

    out.loc[mask, "Face_Amount_Band"] = FACE_CAP_TARGET_BAND
    out.loc[mask, ACTUAL_AMT_COL] = 1_000_000 * out.loc[mask, ACTUAL_CNT_COL]

    return out


def apply_scope_filters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = standardize_frame(df)
    steps = [{"step": "raw_input", "rows": len(out)}]

    if OBSERVATION_YEARS is not None:
        out = out[out["Observation_Year"].isin(OBSERVATION_YEARS)].copy()
        steps.append({"step": f"Observation_Year in {OBSERVATION_YEARS}", "rows": len(out)})

    out = out[
        out["Issue_Age"].between(ISSUE_AGE_MIN, ISSUE_AGE_MAX, inclusive="both")
    ].copy()
    steps.append({"step": f"Issue_Age between {ISSUE_AGE_MIN} and {ISSUE_AGE_MAX}", "rows": len(out)})

    if ATTAINED_AGE_MAX is not None:
        out = out[out["Attained_Age"] <= ATTAINED_AGE_MAX].copy()
        steps.append({"step": f"Attained_Age <= {ATTAINED_AGE_MAX}", "rows": len(out)})

    if SEX_KEEP is not None:
        out = out[out["Sex"].isin(SEX_KEEP)].copy()
        steps.append({"step": f"Sex in {SEX_KEEP}", "rows": len(out)})

    if SMOKER_STATUS_KEEP is not None:
        out = out[out["Smoker_Status"].isin(SMOKER_STATUS_KEEP)].copy()
        steps.append({"step": f"Smoker_Status in {SMOKER_STATUS_KEEP}", "rows": len(out)})

    if INSURANCE_PLAN_DROP:
        out = out[~out["Insurance_Plan"].isin(INSURANCE_PLAN_DROP)].copy()
        steps.append({"step": f"Insurance_Plan not in {INSURANCE_PLAN_DROP}", "rows": len(out)})

    if EXCLUDE_POST_LEVEL_TERM:
        out = out[~out["SOA_Post_Lvl_Ind"].isin(POST_LEVEL_TERM_EXCLUDE_VALUES)].copy()
        steps.append({"step": f"SOA_Post_Lvl_Ind not in {POST_LEVEL_TERM_EXCLUDE_VALUES}", "rows": len(out)})

    if COLLAPSE_DURATION_26_PLUS:
        out["Duration"] = np.where(out["Duration"] >= 26, 26, out["Duration"])
        steps.append({"step": "Duration >= 26 collapsed to 26", "rows": len(out)})

    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    steps.append({"step": f"{EXPECTED_CNT_COL} > 0", "rows": len(out)})

    out = apply_face_amount_cap_1m(out)
    if CAP_FACE_AT_1M:
        steps.append({"step": "Applied 1M face cap rule", "rows": len(out)})

    return out.reset_index(drop=True), pd.DataFrame(steps)


def overall_metrics(df: pd.DataFrame) -> pd.DataFrame:
    actual_cnt = df[ACTUAL_CNT_COL].sum()
    expected_cnt = df[EXPECTED_CNT_COL].sum()
    actual_amt = df[ACTUAL_AMT_COL].sum()
    expected_amt = df[EXPECTED_AMT_COL].sum()

    out = pd.DataFrame({
        "Metric": [
            "Rows",
            "Policies Exposed",
            "Amount Exposed",
            "Death Count",
            "Expected Death Count",
            "A/E Count",
            "Death Claim Amount",
            "Expected Death Amount",
            "A/E Amount",
        ],
        "Value": [
            len(df),
            df["Policies_Exposed"].sum(),
            df["Amount_Exposed"].sum(),
            actual_cnt,
            expected_cnt,
            actual_cnt / expected_cnt if expected_cnt > 0 else np.nan,
            actual_amt,
            expected_amt,
            actual_amt / expected_amt if expected_amt > 0 else np.nan,
        ]
    })
    return out


def ae_summary(df: pd.DataFrame, group_cols: str | list[str]) -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
        ]
        .sum()
        .reset_index()
    )
    out["A/E_Count"] = out[ACTUAL_CNT_COL] / out[EXPECTED_CNT_COL]
    out["A/E_Amount"] = out[ACTUAL_AMT_COL] / out[EXPECTED_AMT_COL]
    return out.sort_values(group_cols).reset_index(drop=True)


def safe_poisson_deviance(y_true: pd.Series, y_pred: pd.Series) -> float:
    y = np.asarray(y_true, dtype=float)
    mu = np.clip(np.asarray(y_pred, dtype=float), 1e-12, None)

    term = np.where(y == 0, 0.0, y * np.log(y / mu))
    return float(2.0 * np.sum(term - (y - mu)))


def weighted_ae_error(actual: pd.Series, predicted: pd.Series, expected: pd.Series) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    expected = np.asarray(expected, dtype=float)

    actual_ae = actual / np.clip(expected, 1e-12, None)
    pred_ae = predicted / np.clip(expected, 1e-12, None)
    return float(np.average(np.abs(actual_ae - pred_ae), weights=np.clip(expected, 1e-12, None)))


def prepare_glm_frame(df: pd.DataFrame, required_cols: list[str]) -> pd.DataFrame:
    out = df[required_cols].copy()

    for col in required_cols:
        if col in CATEGORICAL_COLS:
            out[col] = out[col].astype("string").fillna("Missing")
        elif col in NUMERIC_COLS:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=[c for c in required_cols if c != ACTUAL_AMT_COL and c != EXPECTED_AMT_COL]).copy()
    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    return out


def aggregate_cells(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    metric_cols = [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed"
    ]
    agg = (
        df.groupby(features, dropna=False)[metric_cols]
        .sum()
        .reset_index()
    )
    return agg


def screen_formula_for_feature(feature: str) -> str:
    if feature in NUMERIC_SCREEN_FEATURES:
        return f"{ACTUAL_CNT_COL} ~ bs({feature}, df={SCREEN_SPLINE_DF}, degree=3, include_intercept=False)"
    return f"{ACTUAL_CNT_COL} ~ C({feature})"


def run_univariate_screen(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []

    for feature in features:
        needed = [feature, ACTUAL_CNT_COL, EXPECTED_CNT_COL]
        sub = prepare_glm_frame(df, needed)

        try:
            sub = aggregate_cells(sub, [feature])

            base_model = smf.glm(
                formula=f"{ACTUAL_CNT_COL} ~ 1",
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub[EXPECTED_CNT_COL]),
            ).fit()

            model = smf.glm(
                formula=screen_formula_for_feature(feature),
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub[EXPECTED_CNT_COL]),
            ).fit()

            pred = model.predict(sub, offset=np.log(sub[EXPECTED_CNT_COL]))
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": int(sub[feature].nunique(dropna=False)),
                "Base_Deviance": base_model.deviance,
                "Model_Deviance": model.deviance,
                "Deviance_Reduction": base_model.deviance - model.deviance,
                "Deviance_Reduction_%": 100 * (base_model.deviance - model.deviance) / base_model.deviance if base_model.deviance > 0 else np.nan,
                "AIC": model.aic,
                "Weighted_AE_Error": weighted_ae_error(sub[ACTUAL_CNT_COL], pred, sub[EXPECTED_CNT_COL]),
            })
        except Exception as exc:
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": np.nan,
                "Base_Deviance": np.nan,
                "Model_Deviance": np.nan,
                "Deviance_Reduction": np.nan,
                "Deviance_Reduction_%": np.nan,
                "AIC": np.nan,
                "Weighted_AE_Error": np.nan,
                "Note": str(exc)[:180],
            })

    out = pd.DataFrame(rows)
    sort_cols = ["Deviance_Reduction_%", "Deviance_Reduction"]
    return out.sort_values(sort_cols, ascending=False, na_position="last").reset_index(drop=True)


def fit_poisson_glm(train_df: pd.DataFrame, formula: str):
    model = smf.glm(
        formula=formula,
        data=train_df,
        family=sm.families.Poisson(),
        offset=np.log(train_df[EXPECTED_CNT_COL]),
    )
    return model.fit()


def evaluate_glm(result, df: pd.DataFrame, label: str) -> dict:
    pred = result.predict(df, offset=np.log(df[EXPECTED_CNT_COL]))
    out = {
        "Dataset": label,
        "Rows": len(df),
        "Actual_Deaths": df[ACTUAL_CNT_COL].sum(),
        "Expected_VBT": df[EXPECTED_CNT_COL].sum(),
        "Predicted_Deaths": pred.sum(),
        "Actual_to_Predicted": df[ACTUAL_CNT_COL].sum() / pred.sum() if pred.sum() > 0 else np.nan,
        "Actual_to_VBT": df[ACTUAL_CNT_COL].sum() / df[EXPECTED_CNT_COL].sum() if df[EXPECTED_CNT_COL].sum() > 0 else np.nan,
        "Poisson_Deviance": safe_poisson_deviance(df[ACTUAL_CNT_COL], pred),
        "Weighted_AE_Error": weighted_ae_error(df[ACTUAL_CNT_COL], pred, df[EXPECTED_CNT_COL]),
    }
    return out


def modal_or_median(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return float(series.median())
    mode = series.mode(dropna=False)
    return mode.iloc[0] if len(mode) else series.iloc[0]


def make_reference_row(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    row = {}
    for col in features:
        row[col] = modal_or_median(df[col])
    row[EXPECTED_CNT_COL] = 1.0
    return pd.DataFrame([row])


def plot_partial_curve(result, scoring_df: pd.DataFrame, features: list[str], variable: str,
                       x_values: list | np.ndarray, title: str):
    ref = make_reference_row(scoring_df, features)
    curve = pd.concat([ref] * len(x_values), ignore_index=True)

    if variable in CATEGORICAL_COLS:
        curve[variable] = list(x_values)
    else:
        curve[variable] = np.asarray(x_values, dtype=float)

    pred = result.predict(curve, offset=np.log(curve[EXPECTED_CNT_COL]))
    rel = pred / np.nanmin(pred)

    plt.figure(figsize=(10, 4))
    plt.plot(x_values, rel, marker="o")
    plt.title(title)
    plt.ylabel("Relative factor (min = 1.0)")
    plt.xlabel(variable)
    plt.grid(alpha=0.3)
    plt.show()


def add_validation_bands(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Issue_Age_Band"] = pd.cut(
        out["Issue_Age"],
        bins=[-0.1, 0, 5, 10, 15, 17],
        labels=["0", "1-5", "6-10", "11-15", "16-17"],
        include_lowest=True,
        right=True,
    )
    out["Attained_Age_Band"] = pd.cut(
        out["Attained_Age"],
        bins=list(range(0, 101, 5)),
        include_lowest=True,
        right=False,
    )
    out["Duration_Band"] = pd.cut(
        np.where(out["Duration"] >= 26, 26, out["Duration"]),
        bins=[-0.1, 1, 5, 10, 15, 20, 26],
        labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21-26+"],
        include_lowest=True,
        right=True,
    )
    return out


def calibration_table(df: pd.DataFrame, group_cols: str | list[str], pred_col: str = "Predicted_Deaths") -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, pred_col]
        ]
        .sum()
        .reset_index()
    )
    out["Actual_to_Predicted"] = out[ACTUAL_CNT_COL] / out[pred_col]
    out["Actual_to_VBT"] = out[ACTUAL_CNT_COL] / out[EXPECTED_CNT_COL]
    out["Predicted_to_VBT"] = out[pred_col] / out[EXPECTED_CNT_COL]
    out["Amount_AE_to_VBT"] = out[ACTUAL_AMT_COL] / out[EXPECTED_AMT_COL]
    return out


def residual_heatmap(df: pd.DataFrame, row_col: str, col_col: str, value_col: str = "Actual_to_Predicted"):
    heat = calibration_table(df, [row_col, col_col]).pivot(index=row_col, columns=col_col, values=value_col)

    plt.figure(figsize=(10, 6))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label=value_col)
    plt.xticks(range(len(heat.columns)), [str(c) for c in heat.columns], rotation=45, ha="right")
    plt.yticks(range(len(heat.index)), [str(i) for i in heat.index])
    plt.title(f"{value_col}: {row_col} × {col_col}")
    plt.tight_layout()
    plt.show()


def export_csv(df: pd.DataFrame, filename: str):
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


In [ ]:
# ============================================================
# LOAD DATA, APPLY SCOPE, AND REVIEW OVERVIEW
# ============================================================

raw_df = read_parquet_data(DATA_PATH, columns=BASE_COLUMNS)
scoped_df, scope_log = apply_scope_filters(raw_df)

print(f"Raw rows   : {len(raw_df):,}")
print(f"Scoped rows: {len(scoped_df):,}")

display(scope_log)
display(overall_metrics(scoped_df))


In [ ]:
# Quick structure and sample rows
print("Columns loaded:")
print(list(scoped_df.columns))

display(scoped_df.head(8))


In [ ]:
# One-way A/E review for the most decision-relevant dimensions
for dim in ["Sex", "Insurance_Plan", "Face_Amount_Band", "Observation_Year"]:
    print(f"\n=== {dim} ===")
    display(ae_summary(scoped_df, dim))


---

# Session 2 · Model screening / feature selection

This section ranks candidate features before you lock in the modeling structure.

The screening method is deliberately transparent:

- for each feature, fit a **univariate Poisson GLM**
- use `log(Expected Deaths)` as the **offset**
- compare its deviance against an intercept-only model

This gives a clean first-pass signal of which variables move mortality materially, before you decide which features deserve smoothing and multivariate modeling.


In [ ]:
# ============================================================
# SESSION 2 — UNIVARIATE SCREENING
# ============================================================

screen_input = scoped_df[SCREEN_FEATURES + [ACTUAL_CNT_COL, EXPECTED_CNT_COL]].copy()
screen_results = run_univariate_screen(screen_input, SCREEN_FEATURES)

display(screen_results)

# Save screening output
export_csv(screen_results, "session2_screen_results.csv")


In [ ]:
# Quick plot of screening strength
plot_df = screen_results.dropna(subset=["Deviance_Reduction_%"]).copy()

plt.figure(figsize=(10, max(4, 0.45 * len(plot_df))))
plt.barh(plot_df["Feature"], plot_df["Deviance_Reduction_%"])
plt.gca().invert_yaxis()
plt.xlabel("Deviance reduction (%) vs intercept-only")
plt.title("Session 2 feature screen")
plt.grid(axis="x", alpha=0.3)
plt.show()


In [ ]:
# ============================================================
# MANUAL SHORTLIST CHECKPOINT
# ============================================================
# Edit SHORTLIST_FEATURES in the control panel if needed, then rerun from Session 2 onward.

print("Current SHORTLIST_FEATURES:")
print(SHORTLIST_FEATURES)

shortlist_ae_review = ae_summary(scoped_df, [col for col in SHORTLIST_FEATURES if col in CATEGORICAL_COLS][:2] or ["Sex"])
display(shortlist_ae_review.head(25))


---

# Session 3 · Model smoothing and model selection

This section moves from screening to actual model building.

### Modeling approach
- target: `Death_Count`
- offset: `log(ExpDth_VBT2015_Cnt)`
- framework: **Poisson GLM**
- smoothing: **B-splines** for numeric mortality drivers such as `Issue_Age`, `Duration`, or `Attained_Age`

### Why this is appropriate here
This keeps the workflow actuarially interpretable:
- categorical relativities stay explicit
- age and duration effects are smoothed instead of left fully jagged
- model choice is compared on a true holdout window when `HOLDOUT_YEARS` is provided


In [ ]:
# ============================================================
# BUILD TRAIN / HOLDOUT SETS
# ============================================================

if HOLDOUT_YEARS is None:
    train_raw = scoped_df.copy()
    holdout_raw = scoped_df.copy()
    print("No holdout years supplied: train and evaluation both use full scoped data.")
else:
    train_raw = scoped_df[~scoped_df["Observation_Year"].isin(HOLDOUT_YEARS)].copy()
    holdout_raw = scoped_df[scoped_df["Observation_Year"].isin(HOLDOUT_YEARS)].copy()
    print(f"Train years  : {sorted(train_raw['Observation_Year'].dropna().unique().tolist())}")
    print(f"Holdout years: {sorted(holdout_raw['Observation_Year'].dropna().unique().tolist())}")

print(f"Train rows  : {len(train_raw):,}")
print(f"Holdout rows: {len(holdout_raw):,}")


In [ ]:
# ============================================================
# FIT CANDIDATE MODELS
# ============================================================

candidate_results = []
fitted_models = {}

for model_key, formula in CANDIDATE_MODELS.items():
    required_features = sorted(set(re.findall(r'\b[A-Za-z_][A-Za-z0-9_]*\b', formula)) & set(BASE_COLUMNS))
    required_cols = required_features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]

    train_model_df = prepare_glm_frame(train_raw, required_cols)
    holdout_model_df = prepare_glm_frame(holdout_raw, required_cols)

    # Aggregate only by features used in the formula to make fitting lighter and cleaner.
    train_cells = aggregate_cells(train_model_df, required_features)
    holdout_cells = aggregate_cells(holdout_model_df, required_features)

    try:
        result = fit_poisson_glm(train_cells, formula)
        fitted_models[model_key] = {
            "result": result,
            "features": required_features,
            "formula": formula,
            "train_cells": train_cells,
            "holdout_cells": holdout_cells,
        }

        train_eval = evaluate_glm(result, train_cells, "Train")
        holdout_eval = evaluate_glm(result, holdout_cells, "Holdout")

        candidate_results.append({
            "Model_Key": model_key,
            "Formula": formula,
            "Train_Deviance": train_eval["Poisson_Deviance"],
            "Holdout_Deviance": holdout_eval["Poisson_Deviance"],
            "Train_Actual_to_Pred": train_eval["Actual_to_Predicted"],
            "Holdout_Actual_to_Pred": holdout_eval["Actual_to_Predicted"],
            "Train_Weighted_AE_Error": train_eval["Weighted_AE_Error"],
            "Holdout_Weighted_AE_Error": holdout_eval["Weighted_AE_Error"],
            "AIC": result.aic,
            "Num_Params": len(result.params),
        })
    except Exception as exc:
        candidate_results.append({
            "Model_Key": model_key,
            "Formula": formula,
            "Train_Deviance": np.nan,
            "Holdout_Deviance": np.nan,
            "Train_Actual_to_Pred": np.nan,
            "Holdout_Actual_to_Pred": np.nan,
            "Train_Weighted_AE_Error": np.nan,
            "Holdout_Weighted_AE_Error": np.nan,
            "AIC": np.nan,
            "Num_Params": np.nan,
            "Note": str(exc)[:200],
        })

model_comparison = pd.DataFrame(candidate_results).sort_values(
    ["Holdout_Deviance", "Holdout_Weighted_AE_Error"], ascending=[True, True], na_position="last"
).reset_index(drop=True)

display(model_comparison)
export_csv(model_comparison, "session3_model_comparison.csv")


In [ ]:
# Review the selected model
print("SELECTED_MODEL_KEY =", SELECTED_MODEL_KEY)

selected_bundle = fitted_models[SELECTED_MODEL_KEY]
selected_result = selected_bundle["result"]
selected_features = selected_bundle["features"]

print("\nSelected formula:")
print(selected_bundle["formula"])

print("\nSelected model coefficients (top 25 by absolute value):")
coef_table = (
    pd.DataFrame({
        "term": selected_result.params.index,
        "coef": selected_result.params.values,
        "abs_coef": np.abs(selected_result.params.values),
    })
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)
display(coef_table.head(25))


In [ ]:
# Smoothed curve review for numeric drivers in the selected model
for numeric_var in ["Issue_Age", "Duration", "Attained_Age"]:
    if numeric_var in selected_features:
        if numeric_var == "Issue_Age":
            x_vals = sorted(train_raw["Issue_Age"].dropna().unique())
        elif numeric_var == "Duration":
            x_vals = sorted(np.unique(np.where(train_raw["Duration"] >= 26, 26, train_raw["Duration"])))
        else:
            x_vals = list(range(int(train_raw["Attained_Age"].min()), int(train_raw["Attained_Age"].max()) + 1))

        plot_partial_curve(
            selected_result,
            prepare_glm_frame(train_raw, selected_features + [EXPECTED_CNT_COL]),
            selected_features,
            numeric_var,
            x_vals,
            title=f"Partial curve review: {numeric_var}",
        )


---

# Session 4 · Post-model testing / validation

This section scores the scoped data with the selected model and checks whether the fit is acceptable in the places that matter most:

- overall calibration
- by year
- by sex
- by insurance plan
- by face amount band
- by age and duration slices

The objective here is not just to find the lowest deviance model. It is to confirm that the selected model is stable, interpretable, and not hiding problems in important segments.


In [ ]:
# ============================================================
# FIT FINAL MODEL ON FULL SCOPED DATA USING THE SELECTED FORMULA
# ============================================================

final_formula = CANDIDATE_MODELS[SELECTED_MODEL_KEY]
final_features = fitted_models[SELECTED_MODEL_KEY]["features"]

final_required_cols = final_features + [
    ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL,
    "Policies_Exposed", "Amount_Exposed"
]

full_model_df = prepare_glm_frame(scoped_df, final_required_cols)
full_cells = aggregate_cells(full_model_df, final_features)

final_result = fit_poisson_glm(full_cells, final_formula)

print("Final model fitted on full scoped data.")
print(final_formula)


In [ ]:
# Score raw scoped data for validation slicing
scored_df = scoped_df.copy()
score_input = prepare_glm_frame(scored_df, final_features + [EXPECTED_CNT_COL, ACTUAL_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL])
scored_df = scored_df.loc[score_input.index].copy()
scored_df["Predicted_Deaths"] = final_result.predict(score_input, offset=np.log(score_input[EXPECTED_CNT_COL]))
scored_df = add_validation_bands(scored_df)

overall_validation = pd.DataFrame([{
    "Rows": len(scored_df),
    "Actual_Deaths": scored_df[ACTUAL_CNT_COL].sum(),
    "Expected_VBT": scored_df[EXPECTED_CNT_COL].sum(),
    "Predicted_Deaths": scored_df["Predicted_Deaths"].sum(),
    "Actual_to_Predicted": scored_df[ACTUAL_CNT_COL].sum() / scored_df["Predicted_Deaths"].sum(),
    "Actual_to_VBT": scored_df[ACTUAL_CNT_COL].sum() / scored_df[EXPECTED_CNT_COL].sum(),
    "Amount_AE_to_VBT": scored_df[ACTUAL_AMT_COL].sum() / scored_df[EXPECTED_AMT_COL].sum(),
}])

display(overall_validation)
export_csv(overall_validation, "session4_overall_validation.csv")


In [ ]:
# Core calibration tables
validation_tables = {
    "by_year": calibration_table(scored_df, "Observation_Year"),
    "by_sex": calibration_table(scored_df, "Sex"),
    "by_plan": calibration_table(scored_df, "Insurance_Plan"),
    "by_face_band": calibration_table(scored_df, "Face_Amount_Band"),
    "by_issue_age_band": calibration_table(scored_df, "Issue_Age_Band"),
    "by_duration_band": calibration_table(scored_df, "Duration_Band"),
}

for name, table in validation_tables.items():
    print(f"\n=== {name} ===")
    display(table)
    export_csv(table, f"session4_{name}.csv")


In [ ]:
# Residual heatmap by attained age × duration band
residual_heatmap(scored_df, "Attained_Age_Band", "Duration_Band", value_col="Actual_to_Predicted")


In [ ]:
# Final reviewer-friendly summary
summary_points = pd.DataFrame({
    "Item": [
        "Source file",
        "Scoped rows",
        "Selected model",
        "Holdout years",
        "1M face cap applied",
        "Overall A/P",
        "Overall A/VBT",
    ],
    "Value": [
        str(DATA_PATH),
        f"{len(scoped_df):,}",
        SELECTED_MODEL_KEY,
        str(HOLDOUT_YEARS),
        str(CAP_FACE_AT_1M),
        round(float(overall_validation.loc[0, "Actual_to_Predicted"]), 6),
        round(float(overall_validation.loc[0, "Actual_to_VBT"]), 6),
    ]
})

display(summary_points)
export_csv(summary_points, "session4_summary_points.csv")


## Suggested working pattern

Use the notebook in this order:

1. change the **control panel**
2. rerun **Session 1**
3. review the **Session 2 screening table**
4. update `SHORTLIST_FEATURES` if needed
5. compare models in **Session 3**
6. set `SELECTED_MODEL_KEY`
7. rerun **Session 4** and review calibration before finalizing any table recommendation

This keeps the workflow clean, reviewable, and easy to defend.
